# Speculator Tests — SpeculativeDecodingTrainer E2E

Unified notebook for all speculator modes and test types. Run via papermill from Go tests.

## Modes (`SPECULATOR_MODE`)
- **DATA_ONLY**: Extract hidden states from verifier model using vLLM sidecar. Outputs Arrow dataset, token_freq.pt, and hidden state safetensors to PVC.
- **TRAIN_ONLY**: Train a draft speculator model from extracted hidden states. Supports checkpoint resume via `resume_from_checkpoint=True`.
- **OFFLINE**: Combined extraction + training in a single TrainJob using an external vLLM server. The notebook deploys vLLM, waits for readiness, then submits OFFLINE jobs. Supports progression tracking, checkpoint resume, and response regeneration.
- **ONLINE**: Training with on-the-fly hidden state generation via a vLLM sidecar managed within the same TrainJob pod. Supports progression tracking, checkpoint resume, sidecar verification, and response regeneration.

## Test types (`TEST_TYPE`)
- **extraction**: Success path — submit job, wait for completion, assert status.
- **failure**: Failure scenarios — submit jobs with bad paths, verify failure detection via SDK APIs. Runs sequentially to avoid GPU contention.

## Pipeline test (Go side)
Two Papermill runs in one pod sharing a PVC:
1. DATA_ONLY extraction (+ idempotency re-run)
2. TRAIN_ONLY training (+ checkpoint resume)
3. OFFLINE extraction + training (+ checkpoint resume + regenerate responses)

## Model and dataset sourcing
- **S3 mode (disconnected)**: Notebook downloads both model and dataset from S3 to PVC. Go test verifies bucket exists before enabling S3 mode. Training pods load from local PVC paths — no internet needed.
- **HF mode (connected)**: No download needed in notebook. SDK downloads model and dataset directly from HuggingFace at runtime.

## TLS
Uses `OPENSHIFT_CA_BUNDLE` (default: in-cluster CA cert). Falls back to `NOTEBOOK_INSECURE_TLS=1` for explicit opt-out. Fails hard if neither is available.

## Key environment variables
| Variable | Description |
|---|---|
| `SPECULATOR_MODE` | DATA_ONLY, TRAIN_ONLY, ONLINE, OFFLINE |
| `TEST_TYPE` | extraction or failure |
| `TRAINING_RUNTIME` | ClusterTrainingRuntime name (required) |
| `DATASET_NAME` | HF dataset name or `pvc://` URI |
| `VERIFIER_MODEL` | HF model name or `pvc://` URI |
| `OUTPUT_DIR` | PVC URI for extraction output |
| `TRAIN_OUTPUT_DIR` | PVC URI for training output |
| `VLLM_GPU_COUNT` | GPUs for vLLM sidecar (DATA_ONLY) |
| `TRAIN_GPU_COUNT` | GPUs for training (TRAIN_ONLY) |
| `TARGET_LAYER_IDS` | Comma-separated layer IDs (e.g. `2,14,25,28`) |
| `MAX_SAMPLES` | Max dataset samples for extraction |
| `REGENERATE_RESPONSES` | `true` to regenerate responses before extraction |
| `ENABLE_PROGRESSION_TRACKING` | `true` to enable progression annotations |
| `TEST_IDEMPOTENCY` | `true` to run idempotency re-run after extraction |
| `HIDDEN_STATES_DTYPE` | Hidden states dtype (default: `bfloat16`) |
| `DATAGEN_CONCURRENCY` | Concurrent data generation workers |
| `AWS_DEFAULT_ENDPOINT` | S3 endpoint (enables S3 mode when set) |
| `MODEL_S3_PREFIX` | S3 prefix for model (default: `models/Qwen3-0.6B`) |
| `DATASET_S3_PREFIX` | S3 prefix for dataset JSONL (default: `datasets/ultrachat.jsonl`) |
| `VLLM_IMAGE` | vLLM container image for OFFLINE mode external server |
| `SPECULATOR_VLLM_IMAGE` | Override vLLM image from Go test (env var) |

In [ ]:
import os
import warnings
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore", message=".*Unverified HTTPS.*")

from kubernetes import client as k8s_client
from kubeflow.trainer import TrainerClient
from kubeflow.common.types import KubernetesBackendConfig

openshift_api_url = os.getenv("OPENSHIFT_API_URL", "")
token = os.getenv("NOTEBOOK_USER_TOKEN", "")
namespace = os.getenv("NOTEBOOK_NAMESPACE", "default")
shared_pvc_name = os.getenv("SHARED_PVC_NAME", "shared-pvc")
speculator_mode = os.getenv("SPECULATOR_MODE", "DATA_ONLY")
test_type = os.getenv("TEST_TYPE", "extraction")

print(f"API: {openshift_api_url}")
print(f"Namespace: {namespace}")
print(f"PVC: {shared_pvc_name}")
print(f"Mode: {speculator_mode}")
print(f"Test type: {test_type}")

cfg = k8s_client.Configuration()
cfg.host = openshift_api_url
ca_bundle = os.getenv("OPENSHIFT_CA_BUNDLE", "/var/run/secrets/kubernetes.io/serviceaccount/ca.crt")
if os.path.exists(ca_bundle):
    cfg.ssl_ca_cert = ca_bundle
    cfg.verify_ssl = True
elif os.getenv("NOTEBOOK_INSECURE_TLS") == "1":
    cfg.verify_ssl = False
else:
    raise RuntimeError(
        f"No CA bundle found at {ca_bundle}; set OPENSHIFT_CA_BUNDLE or NOTEBOOK_INSECURE_TLS=1 to opt out"
    )
cfg.api_key = {"authorization": f"Bearer {token}"}

api_client = k8s_client.ApiClient(cfg)
backend_cfg = KubernetesBackendConfig(client_configuration=api_client.configuration)
client = TrainerClient(backend_cfg)
print("TrainerClient initialized")

In [ ]:
training_runtime_name = os.getenv("TRAINING_RUNTIME")
if not training_runtime_name:
    raise RuntimeError("TRAINING_RUNTIME environment variable is required")

speculator_runtime = client.get_runtime(training_runtime_name)
if speculator_runtime is None:
    raise RuntimeError(f"Required runtime '{training_runtime_name}' not found")
print(f"Got runtime: {speculator_runtime.name}")

In [ ]:
# Model and dataset download from S3
#
# S3 mode (disconnected): download model and dataset from S3 to PVC.
#   Go test verifies bucket exists before enabling S3 mode.
#
# HF mode (connected): no download needed here.
#   Go test sets VERIFIER_MODEL to a HuggingFace ID and DATASET_NAME to "ultrachat".
#   The SDK downloads model and dataset directly from HuggingFace at runtime.
import os

notebook_pvc_path = "/opt/app-root/src"

if test_type == "failure":
    print("Skipping model/dataset download — failure tests do not need real data")
else:
    s3_endpoint = os.getenv("AWS_DEFAULT_ENDPOINT", "")
    s3_access_key = os.getenv("AWS_ACCESS_KEY_ID", "")
    s3_secret_key = os.getenv("AWS_SECRET_ACCESS_KEY", "")
    s3_bucket = os.getenv("AWS_STORAGE_BUCKET", "")
    model_s3_prefix = os.getenv("MODEL_S3_PREFIX", "models/Qwen3-0.6B")
    dataset_s3_prefix = os.getenv("DATASET_S3_PREFIX", "datasets/ultrachat.jsonl")

    use_s3 = bool(s3_endpoint and s3_bucket and s3_access_key and s3_secret_key)

    if use_s3:
        model_local_path = f"{notebook_pvc_path}/models/Qwen3-0.6B"
        dataset_local_path = f"{notebook_pvc_path}/datasets/ultrachat.jsonl"

        print(f"S3 mode: downloading model and dataset to shared PVC")
        print(f"  Endpoint: {s3_endpoint}")
        print(f"  Bucket: {s3_bucket}")

        import s3fs

        endpoint_url = s3_endpoint if s3_endpoint.startswith("http") else f"https://{s3_endpoint}"
        fs = s3fs.S3FileSystem(
            key=s3_access_key,
            secret=s3_secret_key,
            endpoint_url=endpoint_url,
            use_ssl=endpoint_url.startswith("https"),
            config_kwargs={"signature_version": "s3v4"},
            client_kwargs={"verify": False},
        )

        # Download model from S3
        if not os.path.exists(os.path.join(model_local_path, "config.json")):
            os.makedirs(model_local_path, exist_ok=True)
            remote_path = f"{s3_bucket}/{model_s3_prefix}"
            count = 0
            for remote_file in fs.find(remote_path):
                rel_path = remote_file[len(remote_path):].lstrip("/")
                if not rel_path:
                    continue
                local_file = os.path.join(model_local_path, rel_path)
                os.makedirs(os.path.dirname(local_file), exist_ok=True)
                fs.get(remote_file, local_file)
                count += 1
            print(f"  Downloaded {count} model files to {model_local_path}")
        else:
            print(f"  Model already exists at {model_local_path}, skipping")

        # Download dataset JSONL from S3
        if not os.path.exists(dataset_local_path):
            os.makedirs(os.path.dirname(dataset_local_path), exist_ok=True)
            remote_file = f"{s3_bucket}/{dataset_s3_prefix}"
            fs.get(remote_file, dataset_local_path)
            print(f"  Downloaded dataset to {dataset_local_path}")

        print("S3 download complete")
    else:
        print("HF mode: SDK downloads model and dataset from HuggingFace at runtime")

In [ ]:
import time

from kubeflow.trainer.rhai.speculator import (
    SpeculativeDecodingTrainer,
    SpeculatorMode,
    SpeculatorType,
    SpeculatorConfig,
)
from kubeflow.trainer.options.common import Name

FAILURE_EVENT_REASONS = {"BackOff", "CrashLoopBackOff", "Failed", "OOMKilled", "OOMKilling", "Killing"}


def check_job_failure(client, job_name):
    """Check if a TrainJob has failed using SDK APIs."""
    failure_confirmed = False
    details = []

    try:
        job = client.get_job(name=job_name)
        if job.status == "Failed":
            failure_confirmed = True
            details.append("job.status=Failed")
    except Exception as e:
        print(f"  get_job() error: {e}")

    try:
        events = client.get_job_events(name=job_name)
        for event in events:
            reason = getattr(event, "reason", "") or ""
            if reason in FAILURE_EVENT_REASONS:
                failure_confirmed = True
                message = getattr(event, "message", "") or ""
                details.append(f"event: reason={reason} message={message[:100]}")
    except Exception as e:
        print(f"  get_job_events() error: {e}")

    return failure_confirmed, details


def run_speculator_failure_scenario(client, scenario_name, trainer_kwargs, runtime,
                                    expect_sdk_error=False, expected_error=None,
                                    failure_timeout=300):
    """Submit a speculator TrainJob expected to fail, verify the failure.

    If expect_sdk_error=True, the SDK should raise before creating a job.
    Otherwise, the job is created and should fail — verified via get_job()/get_job_events().
    Runs ONE job at a time and cleans up before returning.
    """
    print(f"\n{'='*60}")
    print(f"Scenario: {scenario_name}")
    if expect_sdk_error:
        print(f"Expected: SDK validation error (no job created)")
    else:
        print(f"Expected: Job failure detected via get_job()/get_job_events()")
    print(f"{'='*60}")

    job_name = None

    try:
        job_name = client.train(
            trainer=SpeculativeDecodingTrainer(**trainer_kwargs),
            runtime=runtime,
        )

        if expect_sdk_error:
            print(f"FAILED: Expected SDK error but job was created: {job_name}")
            return False

        print(f"TrainJob created: {job_name}")

        # Wait for the job to start running or fail
        try:
            client.wait_for_job_status(name=job_name, status={"Running", "Failed"}, timeout=failure_timeout)
        except Exception as e:
            print(f"Wait for Running/Failed raised: {e}")

        # Poll until failure is confirmed via SDK APIs
        failure_confirmed = False
        deadline = time.time() + failure_timeout

        while time.time() < deadline:
            failure_confirmed, details = check_job_failure(client, job_name)
            if failure_confirmed:
                print(f"Job failure confirmed via SDK: {'; '.join(details)}")
                break
            time.sleep(15)

        if not failure_confirmed:
            print(f"FAILED: Job failure not confirmed within timeout")
            return False

        print(f"PASSED: {scenario_name}")
        return True

    except Exception as e:
        if expect_sdk_error:
            error_msg = f"{type(e).__name__}: {e}"
            print(f"SDK error caught (expected): {error_msg}")
            if expected_error and expected_error not in error_msg:
                print(f"FAILED: Expected error containing '{expected_error}' but got: {error_msg}")
                return False
            print(f"PASSED: {scenario_name}")
            return True
        else:
            print(f"FAILED: {scenario_name} - unexpected error: {e}")
            return False

    finally:
        if job_name:
            try:
                client.delete_job(job_name)
                print(f"Deleted job: {job_name}")
            except Exception as e:
                print(f"Warning: failed to delete job {job_name}: {e}")


print("Helpers loaded")

In [ ]:
# ============================================================
# DATA_ONLY mode
# ============================================================

results = {}

if speculator_mode == "DATA_ONLY":
    vllm_gpu_count = int(os.getenv("VLLM_GPU_COUNT", "1"))
    vllm_resources = {"nvidia.com/gpu": vllm_gpu_count}

    job_name = os.getenv("JOB_NAME", "speculator-extract")

    if test_type == "extraction":
        # --- Success path: submit extraction job, wait for completion ---
        trainer_kwargs = {
            "mode": SpeculatorMode.DATA_ONLY,
            "speculator_type": SpeculatorType.EAGLE3,
            "vllm_resources": vllm_resources,
        }

        verifier_model = os.getenv("VERIFIER_MODEL", "")
        dataset_name = os.getenv("DATASET_NAME", "")

        if verifier_model:
            trainer_kwargs["verifier_model"] = verifier_model
        if dataset_name:
            trainer_kwargs["dataset_name"] = dataset_name

        # Enable regenerate_responses if using HF dataset (not pvc://)
        if dataset_name and not dataset_name.startswith("pvc://"):
            trainer_kwargs["regenerate_responses"] = True

        output_dir = os.getenv("OUTPUT_DIR", "")
        if output_dir:
            trainer_kwargs["output_dir"] = output_dir

        max_samples = os.getenv("MAX_SAMPLES", "")
        if max_samples:
            trainer_kwargs["max_samples"] = int(max_samples)

        enable_progression = os.getenv("ENABLE_PROGRESSION_TRACKING", "")
        if enable_progression:
            trainer_kwargs["enable_progression_tracking"] = enable_progression.lower() == "true"

        regenerate_responses = os.getenv("REGENERATE_RESPONSES", "")
        if regenerate_responses and regenerate_responses.lower() == "true":
            trainer_kwargs["regenerate_responses"] = True

        config_overrides = {}
        target_layer_ids = os.getenv("TARGET_LAYER_IDS", "")
        if target_layer_ids:
            config_overrides["target_layer_ids"] = [int(x) for x in target_layer_ids.split(",")]
        datagen_concurrency = os.getenv("DATAGEN_CONCURRENCY", "")
        if datagen_concurrency:
            config_overrides["datagen_concurrency"] = int(datagen_concurrency)
        hidden_states_dtype = os.getenv("HIDDEN_STATES_DTYPE", "")
        if hidden_states_dtype:
            config_overrides["hidden_states_dtype"] = hidden_states_dtype
        if config_overrides:
            trainer_kwargs["config"] = SpeculatorConfig(**config_overrides)

        trainer_kwargs["packages_to_install"] = ["speculators==0.6.0", "torchvision==0.24.0"]
        print(f"Trainer kwargs: {trainer_kwargs}")

        submitted_name = client.train(
            trainer=SpeculativeDecodingTrainer(**trainer_kwargs),
            runtime=speculator_runtime,
            options=[Name(name=job_name)],
        )
        print(f"TRAINJOB_NAME: {submitted_name}")

        client.wait_for_job_status(name=submitted_name, status={"Running"}, timeout=600)
        client.wait_for_job_status(name=submitted_name, status={"Complete", "Failed"}, timeout=3600)

        job = client.get_job(name=submitted_name)
        print(f"Job status: {job.status}")

        if job.status != "Complete":
            raise RuntimeError(f"DATA_ONLY job failed with status: {job.status}")

        # Verify output artifacts on PVC
        if output_dir and output_dir.startswith("pvc://"):
            pvc_rel_path = output_dir.split("/", 3)[-1] if output_dir.count("/") >= 3 else ""
            local_output_path = os.path.join(notebook_pvc_path, pvc_rel_path)
            print(f"Verifying output artifacts at: {local_output_path}")

            import glob

            arrow_files = glob.glob(os.path.join(local_output_path, "**/*.arrow"), recursive=True)
            token_freq = os.path.join(local_output_path, "token_freq.pt")
            safetensor_files = glob.glob(os.path.join(local_output_path, "hidden_states", "**/*.safetensors"), recursive=True)

            missing = []
            if not arrow_files:
                missing.append("Arrow dataset files")
            if not os.path.exists(token_freq):
                missing.append("token_freq.pt")
            if not safetensor_files:
                missing.append("hidden state safetensors")

            if missing:
                raise RuntimeError(f"DATA_ONLY output artifacts missing: {', '.join(missing)} at {local_output_path}")

            print(f"  Arrow files: {len(arrow_files)}")
            print(f"  token_freq.pt: exists")
            print(f"  Safetensor files: {len(safetensor_files)}")
            print("Output artifact verification: PASSED")

        if os.getenv("TEST_IDEMPOTENCY", "").lower() == "true" and output_dir:
            idempotency_name = f"{job_name}-idempotency"
            print(f"Idempotency test: submitting second job '{idempotency_name}' to same output_dir")
            idempotency_submitted = client.train(
                trainer=SpeculativeDecodingTrainer(**trainer_kwargs),
                runtime=speculator_runtime,
                options=[Name(name=idempotency_name)],
            )
            print(f"IDEMPOTENCY_TRAINJOB_NAME: {idempotency_submitted}")

            client.wait_for_job_status(name=idempotency_submitted, status={"Running"}, timeout=600)
            client.wait_for_job_status(name=idempotency_submitted, status={"Complete", "Failed"}, timeout=3600)

            idempotency_job = client.get_job(name=idempotency_submitted)
            print(f"Idempotency job status: {idempotency_job.status}")

            if idempotency_job.status != "Complete":
                raise RuntimeError(f"Idempotency job failed with status: {idempotency_job.status}")

else:
    print(f"Skipping — mode {speculator_mode} is not DATA_ONLY")

In [ ]:
# ============================================================
# TRAIN_ONLY mode
# ============================================================

if speculator_mode == "TRAIN_ONLY":
    results = {}

    train_gpu_count = int(os.getenv("TRAIN_GPU_COUNT", "1"))
    training_resources = {"nvidia.com/gpu": train_gpu_count}

    verifier_model = os.getenv("VERIFIER_MODEL", "")

    extract_output = os.getenv("OUTPUT_DIR", f"pvc://{shared_pvc_name}/speculator-output/extract")
    hidden_states_path = f"{extract_output}/hidden_states"
    data_path = extract_output
    train_output = os.getenv("TRAIN_OUTPUT_DIR", f"pvc://{shared_pvc_name}/speculator-output/train")

    target_layer_ids = None
    target_layer_ids_str = os.getenv("TARGET_LAYER_IDS", "")
    if target_layer_ids_str:
        target_layer_ids = [int(x) for x in target_layer_ids_str.split(",")]

    hidden_states_dtype = os.getenv("HIDDEN_STATES_DTYPE", "bfloat16")

    if test_type == "extraction":
        trainer_kwargs = {
            "mode": SpeculatorMode.TRAIN_ONLY,
            "speculator_type": SpeculatorType.EAGLE3,
            "training_resources": training_resources,
            "verifier_model": verifier_model,
            "hidden_states_path": hidden_states_path,
            "data_path": data_path,
            "output_dir": train_output,
            "epochs": 3,
            "lr": 1e-4,
        }

        config_overrides = {
            "checkpoint_freq": 1.0,
        }
        if target_layer_ids:
            config_overrides["target_layer_ids"] = target_layer_ids
        if hidden_states_dtype:
            config_overrides["hidden_states_dtype"] = hidden_states_dtype
        trainer_kwargs["config"] = SpeculatorConfig(**config_overrides)


        # === Step 1: Submit TRAIN_ONLY job, interrupt after first checkpoint ===
        print(f"\n{'='*60}")
        print(f"Step 1: Submit TRAIN_ONLY job (epochs=3), interrupt after checkpoint")
        print(f"{'='*60}")
        trainer_kwargs["packages_to_install"] = ["speculators==0.6.0", "torchvision==0.24.0"]
        print(f"TRAIN_ONLY trainer kwargs: {trainer_kwargs}")

        job1_name = client.train(
            trainer=SpeculativeDecodingTrainer(**trainer_kwargs),
            runtime=speculator_runtime,
            options=[Name(name="speculator-train")],
        )
        print(f"TRAIN_ONLY TRAINJOB_NAME: {job1_name}")

        client.wait_for_job_status(name=job1_name, status={"Running"}, timeout=600)
        print("Job is Running, waiting for first checkpoint save...")

        import re as _re_ckpt, time as _time_ckpt
        from kubernetes import client as _k8s_ckpt
        _core_v1 = _k8s_ckpt.CoreV1Api(api_client=api_client)
        _ckpt_pattern = _re_ckpt.compile(r"Checkpoint saved to")
        _deadline = _time_ckpt.time() + 1800
        _ckpt_found = False
        while _time_ckpt.time() < _deadline:
            _all_pods = _core_v1.list_namespaced_pod(namespace)
            for _pod in _all_pods.items:
                if not _pod.metadata.name.startswith(f"{job1_name}-node-"):
                    continue
                if _pod.status.phase == "Running":
                    _logs = _core_v1.read_namespaced_pod_log(
                        _pod.metadata.name, namespace, container="node", tail_lines=200
                    )
                    if _ckpt_pattern.search(_logs):
                        _ckpt_found = True
                        break
            if _ckpt_found:
                break
            _time_ckpt.sleep(3)
        assert _ckpt_found, "No checkpoint was saved within timeout"
        print("First checkpoint detected — waiting for second epoch to start before killing...")

        # Wait for second epoch to start so SIGTERM creates an interrupted checkpoint
        _epoch2_pattern = _re_ckpt.compile(r"Training epoch 2/3 started")
        _epoch2_found = False
        _deadline2 = _time_ckpt.time() + 600
        while _time_ckpt.time() < _deadline2:
            _all_pods = _core_v1.list_namespaced_pod(namespace)
            for _pod in _all_pods.items:
                if not _pod.metadata.name.startswith(f"{job1_name}-node-"):
                    continue
                if _pod.status.phase == "Running":
                    _logs = _core_v1.read_namespaced_pod_log(
                        _pod.metadata.name, namespace, container="node", tail_lines=200
                    )
                    if _epoch2_pattern.search(_logs):
                        _epoch2_found = True
                        break
            if _epoch2_found:
                break
            _time_ckpt.sleep(3)
        assert _epoch2_found, "Second epoch did not start within timeout"
        print("Second epoch started — issuing delete to create interrupted checkpoint")

        client.delete_job(job1_name)
        print(f"Delete issued for: {job1_name}")

        _deadline_del = _time_ckpt.time() + 120
        while _time_ckpt.time() < _deadline_del:
            try:
                client.get_job(job1_name)
                _time_ckpt.sleep(3)
            except Exception:
                break
        print(f"Confirmed TrainJob {job1_name} deleted")

        _deadline_term = _time_ckpt.time() + 120
        while _time_ckpt.time() < _deadline_term:
            _all_pods = _core_v1.list_namespaced_pod(namespace)
            _active = [
                p for p in _all_pods.items
                if p.metadata.name.startswith(f"{job1_name}-node-")
                and p.status.phase in ("Running", "Pending")
            ]
            if not _active:
                break
            print(f"  Waiting for {len(_active)} pod(s) to be cleaned up...")
            _time_ckpt.sleep(5)
        print("Interrupted job fully cleaned up")
        print("Step 1: PASSED (checkpoint saved, job interrupted during epoch 2)")

        # === Step 2: Resume from checkpoint — should find interrupted checkpoint, remove it, resume from epoch 0 ===
        print(f"\n{'='*60}")
        print(f"Step 2: Resume from checkpoint (epochs=3, resume_from_checkpoint=True)")
        print(f"{'='*60}")

        trainer_kwargs["epochs"] = 3

        resume_config = {
            "checkpoint_freq": 1.0,
            "resume_from_checkpoint": True,
        }
        if target_layer_ids:
            resume_config["target_layer_ids"] = target_layer_ids
        if hidden_states_dtype:
            resume_config["hidden_states_dtype"] = hidden_states_dtype
        trainer_kwargs["config"] = SpeculatorConfig(**resume_config)

        job2_name = client.train(
            trainer=SpeculativeDecodingTrainer(**trainer_kwargs),
            runtime=speculator_runtime,
            options=[Name(name="speculator-train-resume")],
        )
        print(f"RESUME TRAINJOB_NAME: {job2_name}")

        client.wait_for_job_status(name=job2_name, status={"Running"}, timeout=600)
        client.wait_for_job_status(name=job2_name, status={"Complete", "Failed"}, timeout=3600)

        job2 = client.get_job(name=job2_name)
        print(f"Resume job status: {job2.status}")

        if job2.status != "Complete":
            raise RuntimeError(f"Resume job failed with status: {job2.status}")

        print("Step 2: PASSED (resumed and completed successfully)")
        print("TRAIN_ONLY interrupt + checkpoint resume: PASSED")

    elif test_type == "failure":
        print("Running TRAIN_ONLY failure scenarios...")

        base_kwargs = {
            "mode": SpeculatorMode.TRAIN_ONLY,
            "speculator_type": SpeculatorType.EAGLE3,
            "training_resources": {"nvidia.com/gpu": 1},
            "verifier_model": f"pvc://{shared_pvc_name}/models/Qwen3-0.6B",
            "epochs": 1,
            "lr": 1e-4,
        }
        if target_layer_ids:
            base_kwargs["config"] = SpeculatorConfig(target_layer_ids=target_layer_ids)

        # Scenario: Incomplete data extraction marker
        # Simulate interrupted DATA_ONLY by writing the SDK's incomplete marker file.
        # TRAIN_ONLY checks for this marker at startup and raises RuntimeError if found.
        incomplete_dir = "/opt/app-root/src/speculator-output/fail-incomplete/hidden_states"
        os.makedirs(incomplete_dir, exist_ok=True)
        marker_path = os.path.join(incomplete_dir, "extraction-is-incomplete.txt.rank-0")
        with open(marker_path, "w") as f:
            f.write("Data extraction in progress (rank 0)")
        print(f"Created incomplete marker at {marker_path}")

        results["Incomplete extraction marker"] = run_speculator_failure_scenario(
            client=client,
            scenario_name="Incomplete extraction marker",
            trainer_kwargs={
                **base_kwargs,
                "hidden_states_path": f"pvc://{shared_pvc_name}/speculator-output/fail-incomplete/hidden_states",
                "data_path": f"pvc://{shared_pvc_name}/speculator-output/fail-incomplete",
                "output_dir": f"pvc://{shared_pvc_name}/speculator-output/fail-incomplete-train",
            },
            runtime=speculator_runtime,
            expect_sdk_error=False,
        )

else:
    print(f"Skipping — mode {speculator_mode} is not TRAIN_ONLY")

In [ ]:
# ============================================================
# ONLINE mode
# ============================================================
# ONLINE performs speculative decoding training with on-the-fly hidden state
# generation via a vLLM sidecar managed within the same TrainJob pod. The SDK
# handles sidecar lifecycle — startup, health check, and shutdown. Hidden states
# are generated per-batch during training and are not persisted to disk.
#
# Test flow: interrupt + resume as the primary E2E path.
# 1. Submit ONLINE job with epochs=3, interrupt after epoch 2 completes
# 2. Resume from checkpoint — completes remaining epoch (3)
# This covers: job creation, progression, config overrides, vLLM sidecar,
#   checkpoint writing, interrupt, resume, full completion, artifact checks.

if speculator_mode == "ONLINE":
    if test_type == "extraction":
        # --- Configuration from env ---
        vllm_gpu_count = int(os.getenv("VLLM_GPU_COUNT", "1"))
        train_gpu_count = int(os.getenv("TRAIN_GPU_COUNT", "1"))
        target_layer_ids_str = os.getenv("TARGET_LAYER_IDS", "2,14,25,28")
        target_layer_ids = [int(x) for x in target_layer_ids_str.split(",")]
        output_dir = os.getenv("OUTPUT_DIR", f"pvc://{shared_pvc_name}/speculator-output/online")
        dataset_name = os.getenv("DATASET_NAME", "ultrachat")
        max_samples = int(os.getenv("MAX_SAMPLES"))
        enable_progression = os.getenv("ENABLE_PROGRESSION_TRACKING", "true").lower() == "true"
        hidden_states_dtype = os.getenv("HIDDEN_STATES_DTYPE", "bfloat16")

        regenerate_responses_env = os.getenv("REGENERATE_RESPONSES", "")
        if regenerate_responses_env:
            regenerate_responses = regenerate_responses_env.lower() == "true"
        else:
            regenerate_responses = not use_s3

        verifier_model_env = os.getenv("VERIFIER_MODEL", "Qwen/Qwen3-0.6B")

        # ONLINE mode: model must be on PVC for the sidecar (pvc:// URI) in disconnected,
        # or a HF ID in connected environments (sidecar downloads at startup).
        if verifier_model_env.startswith("pvc://"):
            verifier_model = verifier_model_env
        else:
            model_name = verifier_model_env.split("/")[-1]
            model_local_path = f"{notebook_pvc_path}/models/{model_name}"
            verifier_model = f"pvc://{shared_pvc_name}/models/{model_name}"
            print(f"ONLINE mode (connected): downloading {verifier_model_env} to PVC for vLLM sidecar")
            os.makedirs(model_local_path, exist_ok=True)
            if not os.path.exists(os.path.join(model_local_path, "config.json")):
                from huggingface_hub import snapshot_download
                snapshot_download(
                    repo_id=verifier_model_env,
                    local_dir=model_local_path,
                    token=os.getenv("HUGGINGFACE_HUB_TOKEN"),
                    resume_download=True,
                    local_dir_use_symlinks=False,
                )
                print(f"  Model downloaded to {model_local_path}")
            else:
                print(f"  Model already exists at {model_local_path}, skipping")

        training_resources = {"nvidia.com/gpu": train_gpu_count, "memory": "64Gi", "cpu": "2"}
        vllm_resources = {"nvidia.com/gpu": vllm_gpu_count}

        print(f"ONLINE config:")
        print(f"  Verifier model: {verifier_model}")
        print(f"  Output dir: {output_dir}")
        print(f"  Dataset: {dataset_name}")
        print(f"  Target layers: {target_layer_ids}")
        print(f"  Train GPUs: {train_gpu_count}")
        print(f"  vLLM sidecar GPUs: {vllm_gpu_count}")
        print(f"  Max samples: {max_samples}")
        print(f"  Regenerate responses: {regenerate_responses}")

        # === Step 1: Submit ONLINE job, interrupt after epoch 2 checkpoint ===
        job_name = "speculator-online"
        trainer_kwargs = {
            "mode": SpeculatorMode.ONLINE,
            "speculator_type": SpeculatorType.EAGLE3,
            "verifier_model": verifier_model,
            "output_dir": output_dir,
            "dataset_name": dataset_name,
            "total_seq_len": 512,
            "max_samples": max_samples,
            "epochs": 3,
            "lr": 3e-5,
            "training_resources": training_resources,
            "vllm_resources": vllm_resources,
            "enable_progression_tracking": enable_progression,
            "regenerate_responses": regenerate_responses,
            "config": SpeculatorConfig(
                target_layer_ids=target_layer_ids,
                hidden_states_dtype=hidden_states_dtype,
                num_layers=1,
                ttt_steps=3,
                norm_before_residual=True,
                scheduler_type="linear",
                checkpoint_freq=1.0,
            ),
        }

        print(f"\n{'='*60}")
        print(f"Step 1: Submit ONLINE job (epochs=3), interrupt after epoch 2")
        print(f"{'='*60}")
        trainer_kwargs["packages_to_install"] = ["speculators==0.6.0", "torchvision==0.24.0"]
        print(f"Trainer kwargs: {trainer_kwargs}")

        submitted = client.train(
            trainer=SpeculativeDecodingTrainer(**trainer_kwargs),
            runtime=speculator_runtime,
            options=[Name(name=job_name)],
        )
        print(f"TRAINJOB_NAME: {submitted}")

        client.wait_for_job_status(name=submitted, status={"Running"}, timeout=600)
        print("Job is Running, waiting for epoch 2 checkpoint to be saved...")

        # Poll pod logs until 2 checkpoints are fully saved (epoch 1 + epoch 2).
        # "Checkpoint saved to" appears AFTER model shards are written to NFS,
        # so no additional flush wait is needed — delete immediately.
        import re as _re_ckpt, time as _time_ckpt
        from kubernetes import client as _k8s_ckpt
        _core_v1 = _k8s_ckpt.CoreV1Api(api_client=api_client)
        _ckpt_saved_pattern = _re_ckpt.compile(r"Checkpoint saved to ")
        _deadline = _time_ckpt.time() + 1800
        _ckpt_found = False
        while _time_ckpt.time() < _deadline:
            _all_pods = _core_v1.list_namespaced_pod(namespace)
            for _pod in _all_pods.items:
                if not _pod.metadata.name.startswith(f"{job_name}-node-"):
                    continue
                if _pod.status.phase == "Running":
                    _logs = _core_v1.read_namespaced_pod_log(
                        _pod.metadata.name, namespace, container="node", tail_lines=200
                    )
                    if len(_ckpt_saved_pattern.findall(_logs)) >= 2:
                        _ckpt_found = True
                        break
            if _ckpt_found:
                break
            _time_ckpt.sleep(3)
        assert _ckpt_found, "Epoch 2 checkpoint was not saved within timeout"
        print("Epoch 2 checkpoint saved to disk — issuing delete")

        # Delete the TrainJob
        client.delete_job(job_name)
        print(f"Delete issued for: {job_name}")

        # Wait for TrainJob to be fully deleted
        _deadline_del = _time_ckpt.time() + 120
        while _time_ckpt.time() < _deadline_del:
            try:
                client.get_job(job_name)
                _time_ckpt.sleep(3)
            except Exception:
                break
        print(f"Confirmed TrainJob {job_name} deleted")

        # Wait for pods to be cleaned up
        _deadline_term = _time_ckpt.time() + 120
        while _time_ckpt.time() < _deadline_term:
            _all_pods = _core_v1.list_namespaced_pod(namespace)
            _active = [
                p for p in _all_pods.items
                if p.metadata.name.startswith(f"{job_name}-node-")
                and p.status.phase in ("Running", "Pending")
            ]
            if not _active:
                break
            print(f"  Waiting for {len(_active)} pod(s) to be cleaned up...")
            _time_ckpt.sleep(5)
        print("Interrupted job fully cleaned up")
        print("Step 1: PASSED (epoch 2 checkpoint saved, job interrupted)")

        # === Step 2: Resume from checkpoint — completes epoch 3 ===
        print(f"\n{'='*60}")
        print(f"Step 2: Resume from checkpoint (completes epoch 3)")
        print(f"{'='*60}")

        resume_name = "speculator-online-resume"
        trainer_kwargs["epochs"] = 3
        trainer_kwargs["regenerate_responses"] = regenerate_responses
        trainer_kwargs["config"] = SpeculatorConfig(
            target_layer_ids=target_layer_ids,
            hidden_states_dtype=hidden_states_dtype,
            num_layers=1,
            ttt_steps=3,
            norm_before_residual=True,
            scheduler_type="linear",
            checkpoint_freq=1.0,
            resume_from_checkpoint=True,
        )

        resume_submitted = client.train(
            trainer=SpeculativeDecodingTrainer(**trainer_kwargs),
            runtime=speculator_runtime,
            options=[Name(name=resume_name)],
        )
        print(f"RESUME TRAINJOB_NAME: {resume_submitted}")

        client.wait_for_job_status(name=resume_submitted, status={"Running"}, timeout=600)
        client.wait_for_job_status(name=resume_submitted, status={"Complete", "Failed"}, timeout=3600)

        resume_job = client.get_job(name=resume_submitted)
        print(f"Resume job status: {resume_job.status}")
        if resume_job.status != "Complete":
            raise RuntimeError(f"Checkpoint resume job failed with status: {resume_job.status}")
        print("Step 2: PASSED (resumed and completed successfully)")

        # === Artifact checks (only when regenerate_responses is off) ===
        if not regenerate_responses:
            _output_rel = output_dir.split("/", 3)[3] if "/" in output_dir else ""
            _output_path = os.path.join(notebook_pvc_path, _output_rel)
            _hs_dir = os.path.join(_output_path, "hidden_states")
            if os.path.exists(_hs_dir):
                _hs_files = [f for f in os.listdir(_hs_dir) if f.endswith(".safetensors")]
                assert len(_hs_files) == 0, (
                    f"ONLINE mode should not persist extraction artifacts, "
                    f"but found {len(_hs_files)} .safetensors files in {_hs_dir}"
                )
                print(f"[Artifact Check] No .safetensors extraction artifacts in {_hs_dir}")
            else:
                print(f"[Artifact Check] hidden_states directory does not exist (expected for ONLINE)")

            _token_freq = os.path.join(_output_path, "token_freq.pt")
            assert not os.path.exists(_token_freq), (
                f"ONLINE mode should not persist token_freq.pt, but found: {_token_freq}"
            )
            print(f"[Artifact Check] No token_freq.pt at {_token_freq}")

            _arrow_files = []
            for _root, _dirs, _files in os.walk(_output_path):
                if os.path.basename(_root) == "data":
                    continue
                _arrow_files.extend(
                    os.path.join(_root, f) for f in _files if f.endswith(".arrow")
                )
            assert len(_arrow_files) == 0, (
                f"ONLINE mode should not persist extraction Arrow files, "
                f"but found {len(_arrow_files)}: {_arrow_files}"
            )
            print(f"[Artifact Check] No extraction Arrow files in {_output_path} (data/ excluded)")
            print("[Artifact Check] PASSED — no extraction artifacts on PVC (on-the-fly confirmed)")
        else:
            print("[Artifact Check] SKIPPED — regenerate_responses=True creates artifacts in connected env")

        if regenerate_responses:
            print("\nregenerate_responses=True included in jobs (connected environment)")
        else:
            print("\nregenerate_responses=False (disconnected environment)")

        print("\nONLINE pipeline: ALL STEPS PASSED")

else:
    print(f"Skipping — mode {speculator_mode} is not ONLINE")

In [ ]:
# ============================================================
# OFFLINE mode
# ============================================================
# OFFLINE performs extraction + training in a single TrainJob by calling an
# externally-deployed vLLM endpoint. The notebook deploys vLLM after the model
# download (Cell 3), waits for readiness, then submits OFFLINE TrainJobs.

if speculator_mode == "OFFLINE":
    import time as _time
    from kubernetes.client import (
        AppsV1Api, V1Deployment, V1DeploymentSpec, V1PodTemplateSpec,
        V1PodSpec, V1Container, V1ContainerPort, V1EnvVar,
        V1ResourceRequirements, V1Volume, V1VolumeMount,
        V1PersistentVolumeClaimVolumeSource, V1LabelSelector,
        V1ObjectMeta, V1Service, V1ServiceSpec, V1ServicePort,
        V1Probe, V1HTTPGetAction,
    )

    apps_v1 = AppsV1Api(api_client)
    core_v1 = k8s_client.CoreV1Api(api_client)

    # --- Configuration from env ---
    vllm_image = os.getenv("VLLM_IMAGE", "")
    if not vllm_image:
        raise RuntimeError("VLLM_IMAGE environment variable is required — Go test extracts it from the ClusterTrainingRuntime")
    train_gpu_count = int(os.getenv("TRAIN_GPU_COUNT", "1"))
    target_layer_ids = None
    target_layer_ids_str = os.getenv("TARGET_LAYER_IDS", "")
    if target_layer_ids_str:
        target_layer_ids = [int(x) for x in target_layer_ids_str.split(",")]
        
    output_dir = os.getenv("OUTPUT_DIR", f"pvc://{shared_pvc_name}/speculator-output/offline")
    hidden_states_path = f"{output_dir}/hidden_states"
    data_path = output_dir
    dataset_name = os.getenv("DATASET_NAME", "ultrachat")
    max_samples = int(os.getenv("MAX_SAMPLES"))
    enable_progression = os.getenv("ENABLE_PROGRESSION_TRACKING", "true").lower() == "true"
    datagen_concurrency = int(os.getenv("DATAGEN_CONCURRENCY", "2"))
    hidden_states_dtype = os.getenv("HIDDEN_STATES_DTYPE", "bfloat16")

    regenerate_responses_env = os.getenv("REGENERATE_RESPONSES", "")
    if regenerate_responses_env:
        regenerate_responses = regenerate_responses_env.lower() == "true"
    else:
        regenerate_responses = not use_s3

    verifier_model_env = os.getenv("VERIFIER_MODEL", "Qwen/Qwen3-0.6B")

    # OFFLINE mode needs the model on PVC for vLLM. If VERIFIER_MODEL is a HF ID
    # (not a pvc:// URI), download it to PVC and convert to pvc:// URI.
    if verifier_model_env.startswith("pvc://"):
        verifier_model = verifier_model_env
    else:
        model_name = verifier_model_env.split("/")[-1]
        model_local_path = f"{notebook_pvc_path}/models/{model_name}"
        verifier_model = f"pvc://{shared_pvc_name}/models/{model_name}"
        print(f"OFFLINE mode (connected): downloading {verifier_model_env} to PVC for vLLM")
        os.makedirs(model_local_path, exist_ok=True)
        if not os.path.exists(os.path.join(model_local_path, "config.json")):
            from huggingface_hub import snapshot_download
            snapshot_download(
                repo_id=verifier_model_env,
                local_dir=model_local_path,
                token=os.getenv("HUGGINGFACE_HUB_TOKEN"),
                resume_download=True,
                local_dir_use_symlinks=False,
            )
            print(f"  Model downloaded to {model_local_path}")
        else:
            print(f"  Model already exists at {model_local_path}, skipping")

    # Compute vLLM mount paths from pvc:// URIs
    # pvc://<name>/some/path → /mnt/some/path (vLLM mounts PVC at /mnt)
    def pvc_uri_to_vllm_path(uri):
        parts = uri.split("/", 3)
        return f"/mnt/{parts[3]}" if len(parts) > 3 else "/mnt"

    vllm_model_path = pvc_uri_to_vllm_path(verifier_model)
    vllm_hs_path = pvc_uri_to_vllm_path(hidden_states_path)

    print(f"OFFLINE config:")
    print(f"  vLLM image: {vllm_image}")
    print(f"  Model (SDK): {verifier_model}")
    print(f"  Model (vLLM): {vllm_model_path}")
    print(f"  Hidden states (SDK): {hidden_states_path}")
    print(f"  Hidden states (vLLM): {vllm_hs_path}")
    print(f"  Output dir: {output_dir}")
    print(f"  Target layers: {target_layer_ids}")
    print(f"  Train GPUs: {train_gpu_count}")
    print(f"  Regenerate responses: {regenerate_responses}")


    # --- Test execution ---
    training_resources = {"nvidia.com/gpu": train_gpu_count, "memory": "64Gi", "cpu": "2"}

    if test_type == "extraction":
        vllm_name = "vllm-server"
        vllm_port = 8234
        vllm_labels = {"app": vllm_name}
        vllm_deployed = False

        deployment = V1Deployment(
            metadata=V1ObjectMeta(name=vllm_name, namespace=namespace),
            spec=V1DeploymentSpec(
                replicas=1,
                selector=V1LabelSelector(match_labels=vllm_labels),
                template=V1PodTemplateSpec(
                    metadata=V1ObjectMeta(labels=vllm_labels),
                    spec=V1PodSpec(
                        containers=[V1Container(
                            name="vllm",
                            image=vllm_image,
                            command=["sh", "-c"],
                            args=[
                                f'python3 -m vllm.entrypoints.cli.main serve '
                                f'"$SPECULATOR_VERIFIER_MODEL" '
                                f'--speculative-config \'{{"method":"extract_hidden_states",'
                                f'"num_speculative_tokens":1,'
                                f'"draft_model_config":{{"hf_config":{{'
                                f'"eagle_aux_hidden_state_layer_ids":[{target_layer_ids_str}]}}}}}}\' '
                                f'--kv-transfer-config \'{{"kv_connector":"ExampleHiddenStatesConnector",'
                                f'"kv_role":"kv_producer",'
                                f'"kv_connector_extra_config":{{'
                                f'"shared_storage_path":"\'$SPECULATOR_HS_PATH\'"}}}}\' '
                                f'--port {vllm_port} '
                                f'--gpu-memory-utilization 0.9 '
                                f'--no-enable-chunked-prefill '
                                f'--trust-remote-code'
                            ],
                            env=[
                                V1EnvVar(name="SPECULATOR_VERIFIER_MODEL", value=vllm_model_path),
                                V1EnvVar(name="SPECULATOR_HS_PATH", value=vllm_hs_path),
                                V1EnvVar(name="HF_HOME", value="/tmp/hf_cache"),
                                V1EnvVar(name="HF_HUB_OFFLINE", value="1"),
                            ],
                            ports=[V1ContainerPort(container_port=vllm_port)],
                            resources=V1ResourceRequirements(
                                limits={"nvidia.com/gpu": "1"},
                                requests={"nvidia.com/gpu": "1"},
                            ),
                            volume_mounts=[V1VolumeMount(name="shared-storage", mount_path="/mnt")],
                            readiness_probe=V1Probe(
                                http_get=V1HTTPGetAction(path="/health", port=vllm_port),
                                initial_delay_seconds=60,
                                period_seconds=10,
                            ),
                        )],
                        volumes=[V1Volume(
                            name="shared-storage",
                            persistent_volume_claim=V1PersistentVolumeClaimVolumeSource(
                                claim_name=shared_pvc_name,
                            ),
                        )],
                    ),
                ),
            ),
        )

        service = V1Service(
            metadata=V1ObjectMeta(name=vllm_name, namespace=namespace),
            spec=V1ServiceSpec(
                selector=vllm_labels,
                ports=[V1ServicePort(port=vllm_port, target_port=vllm_port)],
            ),
        )

        try:
            print(f"Deploying vLLM server...")
            apps_v1.create_namespaced_deployment(namespace=namespace, body=deployment)
            core_v1.create_namespaced_service(namespace=namespace, body=service)
            vllm_deployed = True

            # Wait for vLLM readiness
            vllm_endpoint = f"http://{vllm_name}:{vllm_port}/v1"
            print(f"Waiting for vLLM readiness at {vllm_endpoint}...")
            deadline = _time.time() + 600
            while _time.time() < deadline:
                pods = core_v1.list_namespaced_pod(
                    namespace=namespace, label_selector="app=vllm-server"
                ).items
                if pods and all(
                    c.ready for pod in pods for c in (pod.status.container_statuses or [])
                ):
                    print("vLLM server is ready")
                    break
                _time.sleep(10)
            else:
                raise RuntimeError("vLLM server did not become ready within 600s")

            # === Step 1: Submit OFFLINE job, interrupt after epoch 2 checkpoint ===
            job_name = "speculator-offline"
            trainer_kwargs = {
                "mode": SpeculatorMode.OFFLINE,
                "speculator_type": SpeculatorType.EAGLE3,
                "vllm_endpoint": vllm_endpoint,
                "verifier_model": verifier_model,
                "output_dir": output_dir,
                "hidden_states_path": hidden_states_path,
                "data_path": data_path,
                "dataset_name": dataset_name,
                "total_seq_len": 512,
                "max_samples": max_samples,
                "epochs": 3,
                "lr": 3e-5,
                "training_resources": training_resources,
                "enable_progression_tracking": enable_progression,
                "regenerate_responses": regenerate_responses,
                "config": SpeculatorConfig(
                    target_layer_ids=target_layer_ids,
                    datagen_concurrency=datagen_concurrency,
                    hidden_states_dtype=hidden_states_dtype,
                    num_layers=1,
                    ttt_steps=3,
                    norm_before_residual=True,
                    scheduler_type="linear",
                    checkpoint_freq=1.0,
                ),
            }

            print(f"\n{'='*60}")
            print(f"Step 1: Submit OFFLINE job (epochs=3), interrupt after epoch 2")
            print(f"{'='*60}")
            trainer_kwargs["packages_to_install"] = ["speculators==0.6.0", "torchvision==0.24.0"]
            print(f"Trainer kwargs: {trainer_kwargs}")

            submitted = client.train(
                trainer=SpeculativeDecodingTrainer(**trainer_kwargs),
                runtime=speculator_runtime,
                options=[Name(name=job_name)],
            )
            print(f"TRAINJOB_NAME: {submitted}")

            client.wait_for_job_status(name=submitted, status={"Running"}, timeout=600)
            print("Job is Running, waiting for epoch 2 checkpoint to be saved...")

            # Poll pod logs until 2 checkpoints are fully saved (epoch 1 + epoch 2).
            # "Checkpoint saved to" appears AFTER model shards are written to NFS,
            # so no additional flush wait is needed — delete immediately.
            import re as _re_ckpt, time as _time_ckpt
            from kubernetes import client as _k8s_ckpt
            _core_v1 = _k8s_ckpt.CoreV1Api(api_client=api_client)
            _ckpt_saved_pattern = _re_ckpt.compile(r"Checkpoint saved to ")
            _deadline = _time_ckpt.time() + 1800
            _ckpt_found = False
            while _time_ckpt.time() < _deadline:
                _all_pods = _core_v1.list_namespaced_pod(namespace)
                for _pod in _all_pods.items:
                    if not _pod.metadata.name.startswith(f"{job_name}-node-"):
                        continue
                    if _pod.status.phase == "Running":
                        _logs = _core_v1.read_namespaced_pod_log(
                            _pod.metadata.name, namespace, container="node", tail_lines=200
                        )
                        if len(_ckpt_saved_pattern.findall(_logs)) >= 2:
                            _ckpt_found = True
                            break
                if _ckpt_found:
                    break
                _time_ckpt.sleep(3)
            assert _ckpt_found, "Epoch 2 checkpoint was not saved within timeout"
            print("Epoch 2 checkpoint saved to disk — issuing delete")

            # Delete the TrainJob
            client.delete_job(job_name)
            print(f"Delete issued for: {job_name}")

            # Wait for TrainJob to be fully deleted
            _deadline_del = _time_ckpt.time() + 120
            while _time_ckpt.time() < _deadline_del:
                try:
                    client.get_job(job_name)
                    _time_ckpt.sleep(3)
                except Exception:
                    break
            print(f"Confirmed TrainJob {job_name} deleted")

            # Wait for pods to be cleaned up
            _deadline_term = _time_ckpt.time() + 120
            while _time_ckpt.time() < _deadline_term:
                _all_pods = _core_v1.list_namespaced_pod(namespace)
                _active = [
                    p for p in _all_pods.items
                    if p.metadata.name.startswith(f"{job_name}-node-")
                    and p.status.phase in ("Running", "Pending")
                ]
                if not _active:
                    break
                print(f"  Waiting for {len(_active)} pod(s) to be cleaned up...")
                _time_ckpt.sleep(5)
            print("Interrupted job fully cleaned up")
            print("Step 1: PASSED (epoch 2 checkpoint saved, job interrupted)")

            # === Step 2: Resume from checkpoint — completes epoch 3 ===
            print(f"\n{'='*60}")
            print(f"Step 2: Resume from checkpoint (completes epoch 3)")
            print(f"{'='*60}")

            resume_name = "speculator-offline-resume"
            trainer_kwargs["epochs"] = 3
            trainer_kwargs["regenerate_responses"] = regenerate_responses
            trainer_kwargs["config"] = SpeculatorConfig(
                target_layer_ids=target_layer_ids,
                datagen_concurrency=datagen_concurrency,
                hidden_states_dtype=hidden_states_dtype,
                num_layers=1,
                ttt_steps=3,
                norm_before_residual=True,
                scheduler_type="linear",
                checkpoint_freq=1.0,
                resume_from_checkpoint=True,
            )

            resume_submitted = client.train(
                trainer=SpeculativeDecodingTrainer(**trainer_kwargs),
                runtime=speculator_runtime,
                options=[Name(name=resume_name)],
            )
            print(f"RESUME TRAINJOB_NAME: {resume_submitted}")

            client.wait_for_job_status(name=resume_submitted, status={"Running"}, timeout=600)
            client.wait_for_job_status(name=resume_submitted, status={"Complete", "Failed"}, timeout=3600)

            resume_job = client.get_job(name=resume_submitted)
            print(f"Resume job status: {resume_job.status}")
            if resume_job.status != "Complete":
                raise RuntimeError(f"Checkpoint resume job failed with status: {resume_job.status}")
            print("Step 2: PASSED (resumed and completed successfully)")

            if regenerate_responses:
                print("\nregenerate_responses=True included in jobs (connected environment)")
            else:
                print("\nregenerate_responses=False (disconnected environment)")

            print("\nOFFLINE extraction pipeline: ALL STEPS PASSED")

        finally:
            # Clean up vLLM Deployment + Service to release GPU resources
            if vllm_deployed:
                print("\nCleaning up vLLM server...")
                try:
                    apps_v1.delete_namespaced_deployment(name=vllm_name, namespace=namespace)
                    core_v1.delete_namespaced_service(name=vllm_name, namespace=namespace)
                    print(f"Deleted vLLM Deployment and Service: {vllm_name}")
                except Exception as e:
                    print(f"Warning: failed to delete vLLM resources: {e}")


else:
    print(f"Skipping — mode {speculator_mode} is not OFFLINE")

In [ ]:
# ============================================================
# Summary
# ============================================================

if test_type == "failure" and not results:
    raise RuntimeError(f"No failure scenarios executed for mode {speculator_mode}")

if test_type == "failure":
    print(f"\n{'='*60}")
    print(f"SPECULATOR FAILURE SCENARIOS — {speculator_mode}")
    print(f"{'='*60}")
    for name, passed in results.items():
        status = "PASSED" if passed else "FAILED"
        print(f"  {name}: {status}")
    print(f"{'='*60}")

    if not all(results.values()):
        failed = [name for name, passed in results.items() if not passed]
        raise RuntimeError(f"Failed scenarios: {', '.join(failed)}")
    else:
        print("All scenarios passed")
else:
    print("Notebook execution completed")